# 🦷 DEMO COMPLETO — IA y Seguridad de Datos en Odontología
### Marco Antonio Robles Huamán — Keluma SAC

---

## ▶ SOLO 3 PASOS (+ demo DICOM real al final):

| Paso | Celda | Qué hacer |
|------|-------|-----------|
| 1️⃣ | **SETUP** | Ejecutar una sola vez al abrir |
| 2️⃣ | **MENÚ** | Marcar qué demos quieres correr |
| 3️⃣ | **EJECUTAR TODO** | Presionar ▶ — corre todo solo |
| 4️⃣ | **DEMO DICOM REAL** | Prueba con radiografía DICOM real → predicción IA |

> ⚡ Atajo: **Shift+Enter** para ejecutar cada celda

---
# 1️⃣ SETUP — Ejecutar una sola vez

In [ ]:
print('📦 Instalando librerías...')
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'medmnist', 'tensorflow', 'openpyxl', 'pillow',
    'pydicom', 'scikit-learn', 'seaborn', '--quiet'], check=True)
print('✅ Librerías instaladas')

from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/cursos/curso odontologia/04_outputs'
for sub in ['', '/imagenes_muestra', '/dicom_demo', '/modelo']:
    os.makedirs(BASE + sub, exist_ok=True)

print(f'✅ Google Drive conectado')
print(f'📁 Salida: cursos/curso odontologia/04_outputs/')
print('━'*50)
print('  ✅ SETUP COMPLETO — configura el MENÚ y ejecuta')
print('━'*50)

---
# 2️⃣ MENÚ — Configura qué quieres ejecutar

In [ ]:
#@title 🎛️ MENÚ PRINCIPAL — Marca lo que quieres ejecutar

#@markdown ## ¿Qué módulos ejecutar?
MODULO_1_explorar_dataset       = True  #@param {type:"boolean"}
#@markdown `Módulo 1` — Estadísticas: 112,120 imágenes, 14 enfermedades

MODULO_2_galeria_por_enfermedad = True  #@param {type:"boolean"}
#@markdown `Módulo 2` — Galería: 5 radiografías por enfermedad

MODULO_3_exportar_imagenes      = True  #@param {type:"boolean"}
#@markdown `Módulo 3` — Exportar imágenes PNG/BMP a Google Drive

MODULO_4_demo_dicom             = True  #@param {type:"boolean"}
#@markdown `Módulo 4` — Flujo DICOM: leer → píxeles → anonimizar → PNG

MODULO_5_entrenar_ia            = True  #@param {type:"boolean"}
#@markdown `Módulo 5` — Entrenar red neuronal MobileNetV2 (~4 min)

MODULO_6_gradcam                = True  #@param {type:"boolean"}
#@markdown `Módulo 6` — Mapa de calor Grad-CAM (requiere Módulo 5)

MODULO_7_exportar_excel         = True  #@param {type:"boolean"}
#@markdown `Módulo 7` — Excel: catálogo 112,120 imgs + predicciones

#@markdown ---
#@markdown ## ⚙️ Opciones de exportación de imágenes

N_IMAGENES_MUESTRA = 1000  #@param {type:"slider", min:100, max:3000, step:100}
FORMATO_IMAGEN = "PNG"     #@param ["PNG", "BMP"]
RESOLUCION_PX  = 128       #@param [64, 128, 256]

#@markdown ---
#@markdown ## ⚙️ Opciones del modelo de IA

N_IMAGENES_ENTRENAMIENTO = 3000  #@param {type:"slider", min:500, max:10000, step:500}
N_EPOCHS = 5                     #@param {type:"slider", min:1, max:15, step:1}

print('✅ Configuración guardada')
for nombre, val in [
    ('Módulo 1 - Explorar dataset',   MODULO_1_explorar_dataset),
    ('Módulo 2 - Galería',            MODULO_2_galeria_por_enfermedad),
    ('Módulo 3 - Exportar imágenes',  MODULO_3_exportar_imagenes),
    ('Módulo 4 - Demo DICOM',         MODULO_4_demo_dicom),
    ('Módulo 5 - Entrenar IA',        MODULO_5_entrenar_ia),
    ('Módulo 6 - Grad-CAM',           MODULO_6_gradcam),
    ('Módulo 7 - Excel',              MODULO_7_exportar_excel),
]:
    print(f'   {"✅" if val else "⬜"} {nombre}')
print()
print(f'   Imágenes  : {N_IMAGENES_MUESTRA} · {FORMATO_IMAGEN} · {RESOLUCION_PX}px')
print(f'   IA        : {N_IMAGENES_ENTRENAMIENTO} imgs · {N_EPOCHS} epochs')
print('━'*50)
print('  ▶ Ahora ejecuta la celda EJECUTAR TODO')
print('━'*50)

---
# 3️⃣ EJECUTAR TODO — Una sola celda, corre todo
> Presiona ▶ y espera. No toques nada más.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from medmnist import ChestMNIST
from PIL import Image
from sklearn.metrics import roc_auc_score
import pydicom
import cv2, os, time, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white'})
AZUL='#0D1B40'; CELESTE='#1A73E8'; VERDE='#28a745'; ROJO='#dc3545'

ENFERMEDADES = [
    'Atelectasis','Cardiomegalia','Derrame_pleural','Infiltracion',
    'Masa','Nodulo','Neumonia','Neumotorax',
    'Consolidacion','Edema','Enfisema','Fibrosis',
    'Engrosamiento_pleural','Hernia'
]
NOMBRES_DISPLAY = [
    'Atelectasis','Cardiomegalia','Derrame pleural','Infiltración',
    'Masa','Nódulo','Neumonía','Neumotórax',
    'Consolidación','Edema','Enfisema','Fibrosis',
    'Engros. pleural','Hernia'
]

def guardar(fig, nombre):
    fig.savefig(f'{BASE}/{nombre}', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'   💾 {nombre}')

def titulo_seccion(n, texto):
    print(); print('▓'*60)
    print(f'  MÓDULO {n} — {texto}')
    print('▓'*60)

# ─── CARGA DE DATOS ───────────────────────────────────────────
print('📥 Descargando ChestMNIST...')
ds_train = ChestMNIST(split='train', download=True, size=RESOLUCION_PX)
ds_val   = ChestMNIST(split='val',   download=True, size=RESOLUCION_PX)
ds_test  = ChestMNIST(split='test',  download=True, size=RESOLUCION_PX)

def cargar_split(ds, nombre):
    X, y = [], []
    for i in range(len(ds)):
        img, lbl = ds[i]
        X.append(np.array(img))
        y.append(np.array(lbl).flatten())
    X, y = np.array(X), np.array(y)
    print(f'   ✅ {nombre}: {len(X):,}')
    return X, y

X_train_full, y_train_full = cargar_split(ds_train, 'Train')
X_val_full,   y_val_full   = cargar_split(ds_val,   'Val  ')
X_test_full,  y_test_full  = cargar_split(ds_test,  'Test ')

X_todo  = np.vstack([X_train_full, X_val_full, X_test_full])
y_todo  = np.vstack([y_train_full, y_val_full, y_test_full])
splits  = (['train']*len(X_train_full) + ['val']*len(X_val_full)
           + ['test']*len(X_test_full))
N_TOTAL = len(X_todo)

def nombre_archivo(idx, split, etiqueta):
    enfs = [ENFERMEDADES[i] for i,v in enumerate(etiqueta) if int(v)==1]
    diag = ('Normal' if not enfs
            else '_'.join(enfs[:2]) + (f'_y{len(enfs)-2}mas' if len(enfs)>2 else ''))
    return f'img_{idx+1:05d}_{split}_{diag}.{FORMATO_IMAGEN.lower()}'

NOMBRES = [nombre_archivo(i, splits[i], y_todo[i]) for i in range(N_TOTAL)]
print(f'✅ {N_TOTAL:,} imágenes listas\n')

# ═══ MÓDULO 1 — ESTADÍSTICAS ══════════════════════════════════
if MODULO_1_explorar_dataset:
    titulo_seccion(1, 'EXPLORACIÓN DEL DATASET')
    conteos = y_todo.sum(axis=0).astype(int)
    n_normal = int((y_todo.sum(axis=1)==0).sum())
    n_alguna = N_TOTAL - n_normal
    orden    = np.argsort(conteos)[::-1]

    print(f'  Total      : {N_TOTAL:,}')
    print(f'  Train      : {len(X_train_full):,}')
    print(f'  Val        : {len(X_val_full):,}')
    print(f'  Test       : {len(X_test_full):,}')
    print(f'  Normales   : {n_normal:,} ({n_normal/N_TOTAL*100:.1f}%)')
    print(f'  Hallazgos  : {n_alguna:,} ({n_alguna/N_TOTAL*100:.1f}%)')
    print()
    for i in orden:
        print(f'  {NOMBRES_DISPLAY[i]:<25} {conteos[i]:>7,}  ({conteos[i]/N_TOTAL*100:.1f}%)')

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Dataset: {N_TOTAL:,} radiografías — 14 patologías (NIH ChestMNIST)',
                 fontsize=13, fontweight='bold', color=AZUL)
    enf_ord = [NOMBRES_DISPLAY[i] for i in orden]
    barras  = conteos[orden]
    axes[0].barh(enf_ord, barras,
                 color=[AZUL if c >= np.median(conteos) else CELESTE for c in barras],
                 edgecolor='white', linewidth=1.5)
    for i,(v,p) in enumerate(zip(barras, barras/N_TOTAL*100)):
        axes[0].text(v+200, i, f'{v:,} ({p:.1f}%)', va='center', fontsize=8)
    axes[0].set_title('Casos por patología', fontsize=11)
    axes[0].set_xlim(0, barras.max()*1.35)
    axes[0].grid(axis='x', alpha=0.3)
    axes[1].pie([n_normal, n_alguna],
                labels=[f'Normal\n{n_normal:,} ({n_normal/N_TOTAL*100:.1f}%)',
                        f'Con hallazgo\n{n_alguna:,} ({n_alguna/N_TOTAL*100:.1f}%)'],
                colors=[VERDE, ROJO], startangle=90,
                wedgeprops={'edgecolor':'white','linewidth':3})
    axes[1].set_title('Normal vs Con hallazgo', fontsize=11)
    plt.tight_layout()
    guardar(fig, '01_estadisticas_dataset.png')

# ═══ MÓDULO 2 — GALERÍA ═══════════════════════════════════════
if MODULO_2_galeria_por_enfermedad:
    titulo_seccion(2, 'GALERÍA POR ENFERMEDAD')
    N_COLS = 5
    fig = plt.figure(figsize=(18, 36))
    fig.suptitle(
        f'Galería — Radiografías Reales por Patología\n'
        f'ChestMNIST · NIH · {N_TOTAL:,} imágenes · 14 enfermedades etiquetadas',
        fontsize=14, fontweight='bold', color=AZUL, y=1.005)
    gs = gridspec.GridSpec(15, N_COLS+1, width_ratios=[2]+[1]*N_COLS,
                           hspace=0.06, wspace=0.04)
    filas = list(zip(ENFERMEDADES, NOMBRES_DISPLAY)) + [('NORMAL', 'Normal')]
    for fila, (cod, disp) in enumerate(filas):
        if cod == 'NORMAL':
            idx_pos = np.where(y_train_full.sum(axis=1)==0)[0]
            conteo  = len(idx_pos)
            color_bg, color_txt = '#efffef', VERDE
        else:
            idx_col = ENFERMEDADES.index(cod)
            idx_pos = np.where(y_train_full[:, idx_col]==1)[0]
            conteo  = len(idx_pos)
            color_bg, color_txt = '#eef2ff', AZUL

        ax_lbl = fig.add_subplot(gs[fila, 0])
        ax_lbl.set_facecolor(color_bg)
        ax_lbl.text(0.5, 0.65, disp, ha='center', va='center',
                    fontsize=10, fontweight='bold', color=color_txt)
        ax_lbl.text(0.5, 0.28,
                    f'{conteo:,} casos ({conteo/len(y_train_full)*100:.1f}%)',
                    ha='center', va='center', fontsize=8, color='#555')
        ax_lbl.axis('off')
        for s in ax_lbl.spines.values():
            s.set_visible(True); s.set_edgecolor(CELESTE); s.set_linewidth(1.5)

        muestra = np.random.choice(idx_pos, size=min(5, len(idx_pos)), replace=False)
        for col, img_idx in enumerate(muestra):
            ax = fig.add_subplot(gs[fila, col+1])
            arr = X_train_full[img_idx]
            ax.imshow(arr[:,:,0] if arr.ndim==3 else arr, cmap='gray')
            ax.axis('off')
        for col in range(len(muestra), N_COLS):
            fig.add_subplot(gs[fila, col+1]).axis('off')

    guardar(fig, '02_galeria_por_enfermedad.png')

# ═══ MÓDULO 3 — EXPORTAR IMÁGENES ════════════════════════════
if MODULO_3_exportar_imagenes:
    titulo_seccion(3, f'EXPORTAR {N_IMAGENES_MUESTRA} IMÁGENES')
    print(f'  {FORMATO_IMAGEN} · {RESOLUCION_PX}px · mín 60 por enfermedad')

    sel = set()
    for col in range(14):
        pos = np.where(y_todo[:,col]==1)[0]
        sel.update(np.random.choice(pos, min(60,len(pos)), replace=False).tolist())
    norm_idx = np.where(y_todo.sum(axis=1)==0)[0]
    sel.update(np.random.choice(norm_idx, min(100,len(norm_idx)), replace=False).tolist())
    faltan = N_IMAGENES_MUESTRA - len(sel)
    if faltan > 0:
        resto = list(set(range(N_TOTAL)) - sel)
        sel.update(np.random.choice(resto, min(faltan,len(resto)), replace=False).tolist())
    indices_exp = sorted(sel)[:N_IMAGENES_MUESTRA]

    exportados = []; t0 = time.time()
    for prog, idx in enumerate(indices_exp):
        arr   = X_todo[idx]
        canal = arr[:,:,0] if arr.ndim==3 else arr
        canal = (canal*255).astype(np.uint8) if canal.max()<=1.0 else canal.astype(np.uint8)
        Image.fromarray(canal,'L').resize(
            (RESOLUCION_PX,RESOLUCION_PX), Image.LANCZOS
        ).save(f'{BASE}/imagenes_muestra/{NOMBRES[idx]}', format=FORMATO_IMAGEN)
        exportados.append({'indice':idx+1,'nombre_archivo':NOMBRES[idx],
                           'conjunto':splits[idx],
                           'n_enfermedades':int(y_todo[idx].sum()),
                           'clasificacion':'Normal' if y_todo[idx].sum()==0 else 'Con hallazgo',
                           **{e:int(y_todo[idx][i]) for i,e in enumerate(ENFERMEDADES)}})
        if (prog+1)%200==0:
            print(f'  {prog+1}/{len(indices_exp)} ({time.time()-t0:.0f}s)')

    df_exp = pd.DataFrame(exportados)
    print(f'  ✅ {len(exportados)} imágenes en {time.time()-t0:.0f}s')
    for i,enf in enumerate(ENFERMEDADES):
        print(f'    {NOMBRES_DISPLAY[i]:<25}: {int(df_exp[enf].sum()):>4} exportadas')

# ═══ MÓDULO 4 — DEMO DICOM ═══════════════════════════════════
if MODULO_4_demo_dicom:
    titulo_seccion(4, 'FLUJO DICOM COMPLETO')

    def buscar_dicoms():
        import pydicom.data as pd_data
        data_dir = os.path.dirname(pd_data.__file__)
        encontrados = []
        for root, _, files in os.walk(data_dir):
            for f in files:
                if f.lower().endswith('.dcm'):
                    ruta = os.path.join(root, f)
                    try:
                        dcm = pydicom.dcmread(ruta, stop_before_pixels=True)
                        if hasattr(dcm,'Rows') and hasattr(dcm,'Columns'):
                            encontrados.append(
                                (ruta, f, getattr(dcm,'Modality','N/D')))
                            if len(encontrados)>=3: return encontrados
                    except: pass
        return encontrados

    dicoms = buscar_dicoms()
    print(f'  ✅ {len(dicoms)} archivos DICOM encontrados en pydicom')

    if dicoms:
        fig, axes_d = plt.subplots(len(dicoms), 4, figsize=(18, 5*len(dicoms)))
        if len(dicoms)==1: axes_d = [axes_d]
        fig.suptitle('Flujo DICOM → numpy array → PNG\n'
                     'Así se procesan radiografías médicas reales con Python',
                     fontsize=13, fontweight='bold', color=AZUL)
        for j,h in enumerate(['1️⃣ Metadatos DICOM','2️⃣ Array de píxeles',
                               '3️⃣ Imagen visible','4️⃣ PNG en Drive']):
            axes_d[0][j].set_title(h, fontsize=10, fontweight='bold')

        for fila,(ruta,nombre,modalidad) in enumerate(dicoms):
            try:
                dcm     = pydicom.dcmread(ruta)
                pixeles = dcm.pixel_array
                pmin,pmax = pixeles.min(), pixeles.max()
                p_norm = ((pixeles-pmin)/(pmax-pmin)*255).astype(np.uint8) \
                         if pmax>pmin else pixeles.astype(np.uint8)
                if p_norm.ndim==3: p_norm=p_norm[:,:,0]

                meta = (f'Archivo : {nombre}\n'
                        f'Modalidad: {modalidad}\n\n'
                        f'Filas   : {getattr(dcm,"Rows","N/D")}\n'
                        f'Columnas: {getattr(dcm,"Columns","N/D")}\n'
                        f'Bits    : {getattr(dcm,"BitsStored","N/D")}\n\n'
                        f'Shape   : {pixeles.shape}\n'
                        f'Dtype   : {pixeles.dtype}\n'
                        f'Rango   : [{pmin}–{pmax}]')
                axes_d[fila][0].text(0.05,0.95,meta,
                    transform=axes_d[fila][0].transAxes,
                    fontsize=8,va='top',fontfamily='monospace',
                    bbox=dict(boxstyle='round',facecolor='#f0f4ff',edgecolor=AZUL))
                axes_d[fila][0].axis('off')

                sns.heatmap(p_norm[:8,:8].astype(int), annot=True, fmt='d',
                            cmap='gray', ax=axes_d[fila][1], cbar=False,
                            annot_kws={'size':7})
                axes_d[fila][1].set_title(f'8×8 px de {pixeles.shape}', fontsize=8)

                axes_d[fila][2].imshow(p_norm, cmap='gray')
                axes_d[fila][2].axis('off')

                img_out = Image.fromarray(p_norm,'L').resize((128,128),Image.LANCZOS)
                png_name = f'dicom_muestra_{fila+1}_{os.path.splitext(nombre)[0]}.png'
                img_out.save(f'{BASE}/dicom_demo/{png_name}')
                axes_d[fila][3].imshow(np.array(img_out), cmap='gray')
                axes_d[fila][3].axis('off')
                axes_d[fila][3].set_xlabel(f'Guardado:\n{png_name}', fontsize=8)
                print(f'  ✅ {nombre} → {png_name}')
            except Exception as e:
                print(f'  ⚠️ {nombre}: {e}')
                for c in range(4): axes_d[fila][c].axis('off')

        plt.tight_layout()
        guardar(fig, '03_flujo_dicom_completo.png')

        codigo = '''# CÓDIGO REPLICABLE — DICOM → PNG para radiografías dentales
# Ponente: Marco Antonio Robles Huamán — Keluma SAC
import pydicom, numpy as np
from PIL import Image
import os, pandas as pd

def dicom_a_png(ruta_dcm, ruta_png, resolucion=128):
    dcm = pydicom.dcmread(ruta_dcm)
    px  = dcm.pixel_array
    if px.ndim==3: px=px[:,:,0]
    pmin,pmax = px.min(),px.max()
    p_norm = ((px-pmin)/(pmax-pmin)*255).astype(np.uint8) if pmax>pmin else px.astype(np.uint8)
    Image.fromarray(p_norm,'L').resize((resolucion,resolucion),Image.LANCZOS).save(ruta_png)

def anonimizar_dicom(entrada, salida, id_caso):
    """OBLIGATORIO antes de subir a la nube (HIPAA / Ley 29733 Peru / LGPD Brazil)"""
    dcm = pydicom.dcmread(entrada)
    dcm.PatientName = 'ANONIMO'
    dcm.PatientID   = id_caso
    dcm.PatientBirthDate = ''
    dcm.PatientSex  = ''
    dcm.save_as(salida)
'''
        with open(f'{BASE}/dicom_demo/codigo_dicom_a_png.py','w') as fp:
            fp.write(codigo)
        print(f'  💾 codigo_dicom_a_png.py guardado')

# ═══ MÓDULO 5 — ENTRENAR IA ═══════════════════════════════════
modelo = None
y_pred_prob = None
y_pred = None
y_te   = None
X_te   = None
idx_con = None
idx_sin = None

if MODULO_5_entrenar_ia:
    titulo_seccion(5, 'ENTRENAR RED NEURONAL (Transfer Learning)')

    def preparar(X, y, n):
        lim = min(n, len(X))
        Xn  = X[:lim].astype(np.float32)/255.0
        if   Xn.ndim==3:          Xn = np.stack([Xn]*3, axis=-1)
        elif Xn.shape[-1]==1:     Xn = np.repeat(Xn, 3, axis=-1)
        yn  = (y[:lim].sum(axis=1)>0).astype(int)
        return Xn, yn

    X_tr,y_tr = preparar(X_train_full, y_train_full, N_IMAGENES_ENTRENAMIENTO)
    X_va,y_va = preparar(X_val_full,   y_val_full,   500)
    X_te,y_te = preparar(X_test_full,  y_test_full,  500)

    res     = X_tr.shape[1]
    base_m  = keras.applications.MobileNetV2(
                  input_shape=(res,res,3), include_top=False, weights='imagenet')
    base_m.trainable = False
    modelo  = keras.Sequential([
        base_m,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    modelo.compile(optimizer='adam', loss='binary_crossentropy',
                   metrics=['accuracy', keras.metrics.AUC(name='auc')])
    modelo.build((None, res, res, 3))

    print(f'  Entrenando: {len(X_tr):,} imgs · {N_EPOCHS} epochs...')
    h = modelo.fit(X_tr, y_tr, epochs=N_EPOCHS, batch_size=64,
                   validation_data=(X_va,y_va),
                   callbacks=[keras.callbacks.EarlyStopping(
                       monitor='val_auc', patience=3,
                       restore_best_weights=True, mode='max')],
                   verbose=1)

    modelo.save(f'{BASE}/modelo/modelo_radiografias.keras')
    _, acc, auc_sc = modelo.evaluate(X_te, y_te, verbose=0)
    y_pred_prob = modelo.predict(X_te, verbose=0).flatten()
    y_pred      = (y_pred_prob>0.5).astype(int)
    idx_con     = np.where(y_te==1)[0][:3]
    idx_sin     = np.where(y_te==0)[0][:3]
    print(f'  ✅ Accuracy={acc*100:.1f}% | AUC={auc_sc:.3f}')
    print(f'  💾 modelo_radiografias.keras guardado')

    hist = h.history
    ep_r = range(1, len(hist['accuracy'])+1)
    fig, axes = plt.subplots(1,2,figsize=(14,5))
    fig.suptitle('Curvas de Aprendizaje del Modelo de IA',
                 fontsize=13, fontweight='bold', color=AZUL)
    for ax,(m_tr,m_va,titulo) in zip(axes,[
        ('accuracy','val_accuracy','Accuracy'),
        ('auc','val_auc','AUC')]):
        ax.plot(ep_r, hist[m_tr],'o-',color=CELESTE,lw=2.5,label='Entrenamiento')
        ax.plot(ep_r, hist[m_va],'s--',color=ROJO,lw=2.5,label='Validación')
        ax.set_title(titulo,fontsize=11); ax.legend()
        ax.set_ylim([0.4,1.05]); ax.grid(True,alpha=0.3)
    plt.tight_layout()
    guardar(fig,'04_curvas_aprendizaje.png')

    fig, axes = plt.subplots(2,3,figsize=(14,9))
    fig.suptitle('Predicciones de la IA en Radiografías Nuevas',
                 fontsize=13, fontweight='bold', color=AZUL)
    for pi,di in enumerate(list(idx_con)+list(idx_sin)):
        ax   = axes[pi//3][pi%3]
        prob = y_pred_prob[di]; pred=int(prob>0.5); real=y_te[di]
        ok   = pred==real; col=VERDE if ok else ROJO
        ax.imshow(X_te[di,:,:,0],cmap='gray'); ax.axis('off')
        for s in ax.spines.values(): s.set_visible(True); s.set_edgecolor(col); s.set_linewidth(4)
        ax.set_title(
            f'{"✅" if ok else "❌"} IA: {"HALLAZGO" if pred else "NORMAL"} ({prob*100:.0f}%)\n'
            f'Real: {"HALLAZGO" if real else "NORMAL"}',
            fontsize=9, color=col, fontweight='bold')
    plt.tight_layout()
    guardar(fig,'05_predicciones.png')

# ═══ MÓDULO 6 — GRAD-CAM ══════════════════════════════════════
if MODULO_6_gradcam:
    if modelo is None:
        print('  ⚠️  Grad-CAM requiere Módulo 5 activo. Actívalo y vuelve a ejecutar.')
    else:
        titulo_seccion(6, 'MAPA DE CALOR GRAD-CAM')

        def gradcam(modelo, imagen):
            base_m = modelo.layers[0]
            nombre_capa = None
            for capa in reversed(base_m.layers):
                if isinstance(capa, tf.keras.layers.Conv2D):
                    nombre_capa = capa.name; break
            if not nombre_capa:
                return np.zeros(imagen.shape[:2])
            conv_layer = base_m.get_layer(nombre_capa)
            extractor  = tf.keras.Model(
                inputs=base_m.inputs,
                outputs=[conv_layer.output, base_m.output])
            img_b = tf.cast(np.expand_dims(imagen,0), tf.float32)
            with tf.GradientTape() as tape:
                conv_acts, base_out = extractor(img_b)
                tape.watch(conv_acts)
                x = base_out
                for layer in modelo.layers[1:]:
                    x = layer(x)
                score = x[:,0]
            grads = tape.gradient(score, conv_acts)
            if grads is None:
                return np.zeros(imagen.shape[:2])
            pesos = tf.reduce_mean(grads, axis=(0,1,2))
            hm    = np.dot(conv_acts.numpy()[0], pesos.numpy())
            hm    = np.maximum(hm,0)
            return hm/hm.max() if hm.max()>0 else hm

        muestra_gc = list(idx_con[:2]) + list(idx_sin[:2])
        fig, axes  = plt.subplots(len(muestra_gc), 3, figsize=(12, 4*len(muestra_gc)))
        fig.suptitle('Grad-CAM — Dónde Mira la IA al Diagnosticar\n'
                     'Lo mismo que hace Pearl y Overjet en radiografías dentales',
                     fontsize=13, fontweight='bold', color=AZUL)
        for idx_row, di in enumerate(muestra_gc):
            img  = X_te[di]; prob = y_pred_prob[di]
            hm   = gradcam(modelo, img)
            hm_r = cv2.resize(hm,(img.shape[1],img.shape[0]))
            hm_c = cm.jet(hm_r)[:,:,:3]
            img_rgb = img if img.shape[-1]==3 else np.stack([img[:,:,0]]*3, axis=-1)
            superpuesta = np.clip(0.45*img_rgb + 0.55*hm_c, 0, 1)
            axes[idx_row][0].imshow(img[:,:,0],cmap='gray'); axes[idx_row][0].axis('off')
            axes[idx_row][0].set_title('Original',fontsize=10)
            axes[idx_row][1].imshow(hm_r,cmap='jet');  axes[idx_row][1].axis('off')
            axes[idx_row][1].set_title('Mapa de calor',fontsize=10)
            axes[idx_row][2].imshow(superpuesta);        axes[idx_row][2].axis('off')
            axes[idx_row][2].set_title(
                f'IA: {"HALLAZGO" if prob>0.5 else "NORMAL"} ({prob*100:.0f}%)',
                fontsize=10,
                color=ROJO if prob>0.5 else VERDE, fontweight='bold')
        plt.tight_layout()
        guardar(fig,'06_gradcam_mapa_calor.png')

# ═══ MÓDULO 7 — EXCEL ═════════════════════════════════════════
if MODULO_7_exportar_excel:
    titulo_seccion(7, 'EXPORTAR EXCEL COMPLETO')
    ruta_excel = f'{BASE}/07_catalogo_y_resultados.xlsx'

    with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
        pd.DataFrame({
            'ID':              range(1, N_TOTAL+1),
            'Nombre_archivo':  NOMBRES,
            'Conjunto':        splits,
            'N_enfermedades':  y_todo.sum(axis=1).astype(int),
            'Clasificacion':   ['Normal' if s==0 else 'Con hallazgo'
                                for s in y_todo.sum(axis=1)],
            **{e: y_todo[:,i].astype(int) for i,e in enumerate(ENFERMEDADES)}
        }).to_excel(writer, sheet_name='Catalogo 112120 imagenes', index=False)
        print('  ✅ Hoja 1: Catálogo completo (112,120 filas)')

        cnt = y_todo.sum(axis=0).astype(int)
        pd.DataFrame({
            'Enfermedad':  NOMBRES_DISPLAY,
            'Total':       cnt,
            '% dataset':   np.round(cnt/N_TOTAL*100,2),
            'Train':       y_train_full.sum(axis=0).astype(int),
            'Val':         y_val_full.sum(axis=0).astype(int),
            'Test':        y_test_full.sum(axis=0).astype(int),
        }).sort_values('Total',ascending=False).to_excel(
            writer, sheet_name='Estadisticas', index=False)
        print('  ✅ Hoja 2: Estadísticas por enfermedad')

        if modelo is not None and y_pred is not None:
            pd.DataFrame({
                'ID':             range(1, len(y_te)+1),
                'Real':           ['Hallazgo' if v else 'Normal' for v in y_te],
                'Prediccion_IA':  ['Hallazgo' if v else 'Normal' for v in y_pred],
                'Probabilidad_%': np.round(y_pred_prob*100,1),
                'Correcto':       ['Sí' if r==p else 'No' for r,p in zip(y_te,y_pred)],
                'Tipo':           ['VP' if r==1 and p==1 else
                                   'VN' if r==0 and p==0 else
                                   'FP' if r==0 and p==1 else 'FN'
                                   for r,p in zip(y_te,y_pred)]
            }).to_excel(writer, sheet_name='Predicciones Modelo', index=False)
            print('  ✅ Hoja 3: Predicciones del modelo (500 casos)')

        pd.DataFrame({
            'Término': [
                'NPZ','DICOM','PNG','BMP','Transfer Learning','MobileNetV2',
                'Accuracy','AUC','Precision','Recall',
                'VP','VN','FP','FN','Grad-CAM',
                'Epoch','Batch','Sigmoid','Dropout',
                'HIPAA','Ley 29733','LGPD'
            ],
            'Definición': [
                'NumPy comprimido — todas las imágenes en un solo archivo',
                'Digital Imaging and Communications in Medicine — estándar del equipo médico',
                'Sin pérdida + comprimido — recomendado para IA médica',
                'Sin compresión ni pérdida — archivos grandes, compatible con todo',
                'Reutilizar modelo preentrenado y adaptarlo con pocos datos',
                'Red neuronal ligera de Google preentrenada con 1.4M imágenes',
                'Porcentaje de imágenes clasificadas correctamente',
                'Área bajo la curva ROC — 0.5=azar 1.0=perfecto',
                'De todas las alarmas activadas, qué % eran reales',
                'De todos los hallazgos reales, qué % detectó el modelo',
                'Verdadero Positivo: hallazgo detectado correctamente',
                'Verdadero Negativo: normal clasificada correctamente',
                'Falso Positivo: normal que la IA marcó como hallazgo',
                'Falso Negativo: hallazgo real que la IA NO detectó (el más peligroso)',
                'Mapa visual de dónde mira la IA para decidir',
                'Una pasada completa por todos los datos',
                'Grupo de imágenes procesadas simultáneamente',
                'Convierte cualquier número en probabilidad 0–1',
                'Desactiva neuronas aleatoriamente para generalizar mejor',
                'Health Insurance Portability and Accountability Act — EE.UU.',
                'Ley de Protección de Datos Personales — Perú',
                'Lei Geral de Proteção de Dados — Brasil'
            ]
        }).to_excel(writer, sheet_name='Glosario IA', index=False)
        print('  ✅ Hoja 4: Glosario IA')

    tam = os.path.getsize(ruta_excel)/1024/1024
    print(f'  💾 07_catalogo_y_resultados.xlsx ({tam:.1f} MB)')

# ═══ RESUMEN FINAL ════════════════════════════════════════════
print()
print('▓'*60)
print('  ✅ DEMO COMPLETO — ARCHIVOS EN GOOGLE DRIVE')
print('▓'*60)
for carpeta, etiqueta in [
    (BASE,                      'Raíz'),
    (f'{BASE}/imagenes_muestra','imagenes_muestra/'),
    (f'{BASE}/dicom_demo',      'dicom_demo/'),
    (f'{BASE}/modelo',          'modelo/')
]:
    if os.path.exists(carpeta):
        archivos = [f for f in os.listdir(carpeta) if os.path.isfile(f'{carpeta}/{f}')]
        if archivos:
            print(f'  📁 {etiqueta} ({len(archivos)} archivos)')
            for a in sorted(archivos)[:8]:
                kb = os.path.getsize(f'{carpeta}/{a}')/1024
                ic = ('🖼️' if a.endswith(('.png','.bmp')) else
                      '📗' if a.endswith('.xlsx') else
                      '🧠' if a.endswith('.keras') else '🐍')
                print(f'     {ic} {a:<45} {kb:>6.0f} KB')
            if len(archivos)>8: print(f'     ... y {len(archivos)-8} más')
            print()
print('━'*60)
print('  🎓 Marco Antonio Robles Huamán — Keluma SAC')
print('  ▶ Ejecuta la celda siguiente para probar con DICOM real')
print('━'*60)

---
# 4️⃣ DEMO DICOM REAL — Lo que haría el alumno con sus propias Rx

> Esta celda descarga una radiografía DICOM **real** de internet,
> la procesa paso a paso y la pasa al modelo de IA.
> 
> **Reemplaza la URL** con la ruta a tu propio archivo `.dcm` para usar con radiografías dentales reales.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  DEMO DICOM REAL — Descarga, preprocesa y predice con IA
#  Esto es exactamente lo que haría un alumno con sus propias Rx
# ═══════════════════════════════════════════════════════════════
import urllib.request, pydicom, numpy as np, cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tensorflow import keras
import warnings; warnings.filterwarnings('ignore')

AZUL='#0D1B40'; VERDE='#28a745'; ROJO='#dc3545'; CELESTE='#1A73E8'

# ─── PASO 1: Descargar DICOM real ────────────────────────────
print('PASO 1: Descargando radiografía DICOM real...')
URL_DICOM = 'https://github.com/pydicom/pydicom/raw/main/pydicom/data/test_files/CT_small.dcm'
RUTA_DCM  = '/tmp/radiografia_real.dcm'

urllib.request.urlretrieve(URL_DICOM, RUTA_DCM)
print(f'  ✅ Descargado: {RUTA_DCM}')

# ─── PASO 2: Leer metadatos (info del paciente) ───────────────
print()
print('PASO 2: Leer metadatos DICOM (datos del paciente)...')
dcm = pydicom.dcmread(RUTA_DCM)
print(f'  PatientName     : {getattr(dcm, "PatientName", "N/D")}')
print(f'  PatientID       : {getattr(dcm, "PatientID", "N/D")}')
print(f'  PatientBirthDate: {getattr(dcm, "PatientBirthDate", "N/D")}')
print(f'  Modality        : {getattr(dcm, "Modality", "N/D")}')
print(f'  Rows × Columns  : {getattr(dcm, "Rows", "?")} × {getattr(dcm, "Columns", "?")} px')
print(f'  BitsStored      : {getattr(dcm, "BitsStored", "N/D")} bits')
print(f'  Manufacturer    : {getattr(dcm, "Manufacturer", "N/D")}')
print()
print('  ⚠️  ESTOS DATOS SON PRIVADOS — hay que anonimizar antes de subir a la nube')

# ─── PASO 3: Anonimizar (HIPAA / Ley 29733 / LGPD) ──────────
print()
print('PASO 3: Anonimizar — eliminar datos del paciente...')
RUTA_ANON = '/tmp/radiografia_anonima.dcm'
dcm_anon  = pydicom.dcmread(RUTA_DCM)
dcm_anon.PatientName      = 'ANONIMO'
dcm_anon.PatientID        = 'CASO_001'
dcm_anon.PatientBirthDate = ''
dcm_anon.PatientSex       = ''
if hasattr(dcm_anon, 'OtherPatientIDs'): dcm_anon.OtherPatientIDs = ''
dcm_anon.save_as(RUTA_ANON)
print(f'  ✅ Guardado sin datos personales: {RUTA_ANON}')
print(f'  PatientName ahora: {dcm_anon.PatientName}')

# ─── PASO 4: Preprocesamiento — DICOM → array listo para IA ──
print()
print('PASO 4: Preprocesamiento (Python lo hace solo)...')
px    = dcm.pixel_array
if px.ndim == 3: px = px[:,:,0]          # si tiene canales, tomar el primero
pmin, pmax = px.min(), px.max()

# ① Normalizar 0–255 (DICOM puede tener 0–4095 u otros rangos)
p255  = ((px - pmin) / (pmax - pmin) * 255).astype(np.uint8)
print(f'  ① Rango original : [{pmin} – {pmax}]  →  normalizado a [0 – 255]')

# ② Grayscale → RGB (MobileNetV2 necesita 3 canales)
rgb   = np.stack([p255, p255, p255], axis=-1)
print(f'  ② Shape grayscale: {p255.shape}  →  RGB: {rgb.shape}')

# ③ Resize a la resolución del modelo
RES   = 128  # debe coincidir con RESOLUCION_PX del entrenamiento
img_r = cv2.resize(rgb, (RES, RES))
print(f'  ③ Resize: {rgb.shape[:2]}  →  {img_r.shape[:2]}')

# ④ Normalizar 0.0–1.0 (Keras espera float entre 0 y 1)
img_f = img_r.astype(np.float32) / 255.0
print(f'  ④ Normalizar: [0–255]  →  [0.0–1.0]  dtype: {img_f.dtype}')

# ⑤ Agregar dimensión de batch: (128,128,3) → (1,128,128,3)
img_batch = np.expand_dims(img_f, axis=0)
print(f'  ⑤ Batch dim: {img_f.shape}  →  {img_batch.shape}  ← listo para Keras')

# ─── PASO 5: Predicción con el modelo entrenado ──────────────
print()
print('PASO 5: Predicción con el modelo de IA...')
try:
    # Intentar usar el modelo de la sesión actual
    if modelo is not None:
        prob = float(modelo.predict(img_batch, verbose=0)[0][0])
        fuente_modelo = 'modelo entrenado en esta sesión'
    else:
        # Cargar modelo guardado en Drive
        ruta_keras = f'{BASE}/modelo/modelo_radiografias.keras'
        modelo_cargado = keras.models.load_model(ruta_keras)
        prob = float(modelo_cargado.predict(img_batch, verbose=0)[0][0])
        fuente_modelo = 'modelo cargado desde Drive'

    diagnostico = 'HALLAZGO DETECTADO' if prob > 0.5 else 'NORMAL — Sin hallazgo'
    color_dx    = ROJO if prob > 0.5 else VERDE
    print(f'  Fuente: {fuente_modelo}')
    print(f'  Probabilidad de hallazgo : {prob*100:.1f}%')
    print(f'  Diagnóstico IA           : {diagnostico}')
    hay_modelo = True
except Exception as e:
    print(f'  ⚠️  No hay modelo disponible: {e}')
    print('      Ejecuta primero el Módulo 5 (EJECUTAR TODO con Módulo 5 = True)')
    prob, diagnostico, color_dx, hay_modelo = 0, 'Sin modelo', '#888', False

# ─── VISUALIZACIÓN COMPLETA DEL FLUJO ────────────────────────
print()
print('Generando figura del flujo completo...')
fig = plt.figure(figsize=(18, 10))
fig.suptitle('DEMO DICOM REAL — Flujo Completo de Preprocesamiento y Predicción IA\n'
             'Esto es lo que haría el alumno con sus propias radiografías dentales',
             fontsize=13, fontweight='bold', color=AZUL)

gs = gridspec.GridSpec(2, 5, figure=fig, hspace=0.4, wspace=0.35)

titulos_pasos = [
    '① DICOM original\n(píxeles crudos)',
    '② Normalizado\n0 – 255',
    '③ RGB\n(3 canales)',
    '④ Resize 128×128\n÷255 → 0.0–1.0',
    '⑤ PREDICCIÓN IA'
]
imagenes = [p255, p255, rgb, img_r, img_r]
cmaps    = ['gray','gray','viridis','gray','gray']

for col, (titulo, img_vis, cmap) in enumerate(zip(titulos_pasos, imagenes, cmaps)):
    ax = fig.add_subplot(gs[0, col])
    ax.imshow(img_vis if img_vis.ndim==3 else img_vis, cmap=cmap if img_vis.ndim==2 else None)
    ax.axis('off')
    ax.set_title(titulo, fontsize=9, fontweight='bold')
    if col < 4:
        ax.text(1.08, 0.5, '→', transform=ax.transAxes,
                fontsize=20, color=CELESTE, va='center', ha='center')

# Panel de resultado
ax_res = fig.add_subplot(gs[1, :])
ax_res.axis('off')
ax_res.set_facecolor('#f8f9fa')

# Info metadatos
meta_txt = (
    f'METADATOS DICOM ORIGINAL:\n'
    f'  Modality  : {getattr(dcm, "Modality", "N/D")}\n'
    f'  Tamaño    : {getattr(dcm, "Rows", "?")} × {getattr(dcm, "Columns", "?")} px\n'
    f'  Bits      : {getattr(dcm, "BitsStored", "N/D")} bits\n'
    f'  Rango px  : [{pmin} – {pmax}]\n\n'
    f'TRAS ANONIMIZACIÓN:\n'
    f'  PatientName: ANONIMO\n'
    f'  PatientID  : CASO_001\n'
    f'  Fecha nac. : (eliminada)'
)
ax_res.text(0.02, 0.95, meta_txt, transform=ax_res.transAxes,
            fontsize=9, va='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='#eef2ff', edgecolor=AZUL))

# Resultado IA
if hay_modelo:
    res_txt = (
        f'RESULTADO DEL MODELO DE IA\n\n'
        f'  Probabilidad de hallazgo:\n\n'
        f'  {prob*100:.1f}%\n\n'
        f'  {diagnostico}'
    )
    ax_res.text(0.5, 0.95, res_txt, transform=ax_res.transAxes,
                fontsize=11, va='top', ha='center', fontweight='bold',
                color=color_dx,
                bbox=dict(boxstyle='round', facecolor='#fff9f0', edgecolor=color_dx, linewidth=2))

# Código replicable
codigo_txt = (
    'CÓDIGO PARA TUS PROPIAS Rx DENTALES:\n\n'
    'px  = pydicom.dcmread("tu_rx.dcm").pixel_array\n'
    'p255= ((px-px.min())/(px.max()-px.min())*255).astype(np.uint8)\n'
    'rgb = np.stack([p255,p255,p255], axis=-1)\n'
    'img = cv2.resize(rgb,(128,128)).astype(np.float32)/255.0\n'
    'prob= modelo.predict(np.expand_dims(img,0))[0][0]\n'
    'print(f"Hallazgo: {prob*100:.1f}%")'
)
ax_res.text(0.98, 0.95, codigo_txt, transform=ax_res.transAxes,
            fontsize=8, va='top', ha='right', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='#1e1e2e', edgecolor='#444'))
# color del código
ax_res.text(0.98, 0.95, codigo_txt, transform=ax_res.transAxes,
            fontsize=8, va='top', ha='right', fontfamily='monospace', color='#89b4fa',
            bbox=dict(boxstyle='round', facecolor='#1e1e2e', edgecolor='#555'))

plt.tight_layout()
ruta_fig = f'{BASE}/08_demo_dicom_real_completo.png'
fig.savefig(ruta_fig, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'  💾 08_demo_dicom_real_completo.png guardado en Drive')
print()
print('━'*60)
print('  ✅ DEMO COMPLETO')
print()
print('  Para usar con TUS propias radiografías dentales:')
print('  1. Sube tu archivo .dcm a Colab (ícono carpeta → subir)')
print('  2. Cambia URL_DICOM por la ruta local: /content/tu_rx.dcm')
print('  3. Ejecuta esta celda de nuevo')
print('━'*60)